
<a href="https://www.zero-grad.com/">
         <img alt="Zero Grad" src="https://i.postimg.cc/vBPDms4J/pythonml-2.png" >
      </a>

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.templates.default = "presentation"

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')


In [3]:
# Load the dataset
df = pd.read_csv('diabetes_clean.csv')
df.head()

,pregnancies,glucose,diastolic,triceps,insulin,bmi,dpf,age,diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   pregnancies  768 non-null    int64  
 1   glucose      768 non-null    int64  
 2   diastolic    768 non-null    int64  
 3   triceps      768 non-null    int64  
 4   insulin      768 non-null    int64  
 5   bmi          768 non-null    float64
 6   dpf          768 non-null    float64
 7   age          768 non-null    int64  
 8   diabetes     768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [15]:
df.describe()

,pregnancies,glucose,diastolic,triceps,insulin,bmi,dpf,age,diabetes
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [6]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
sc = RobustScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [19]:
# Create a KNN classifier
knn = KNeighborsClassifier()

# Cross validate the model
cv_scores = cross_val_score(knn, X_train, y_train, cv=5)

# Print the scores and mean
print(cv_scores)
print('cv_scores mean:{}'.format(np.mean(cv_scores)))

[0.70731707 0.76422764 0.77235772 0.77235772 0.75409836]
cv_scores mean:0.7540717046514728


In [20]:
# Define the parameter grid to search over
param_grid = {'n_neighbors': range(1, 21, 2), 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}

# Create a KNN classifier
knn = KNeighborsClassifier()

# Perform grid search over the parameter grid using 5-fold cross-validation
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Print the best parameter and best score found by grid search
print("Best parameter:", grid_search.best_params_)
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))
print("Train set score: {:.2f}".format(grid_search.score(X_train, y_train)))
print("Test set score: {:.2f}".format(grid_search.score(X_test, y_test)))

Best parameter: {'metric': 'euclidean', 'n_neighbors': 17, 'weights': 'uniform'}
Best cross-validation score: 0.77
Train set score: 0.79
Test set score: 0.72


In [21]:
# Define the parameter grid to search over
param_grid = {'n_neighbors': range(1, 21, 2)}

# Create a KNN classifier
knn = KNeighborsClassifier()

# Perform grid search over the parameter grid using 5-fold cross-validation
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Print the best parameter and best score found by grid search
print("Best parameter:", grid_search.best_params_)
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))
print("Train set score: {:.2f}".format(grid_search.score(X_train, y_train)))
print("Test set score: {:.2f}".format(grid_search.score(X_test, y_test)))

Best parameter: {'n_neighbors': 17}
Best cross-validation score: 0.77
Train set score: 0.79
Test set score: 0.72


In [11]:
grid = pd.DataFrame(grid_search.cv_results_)
grid.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002716,0.000977,0.009528,0.002338,1,{'n_neighbors': 1},0.682927,0.764228,0.666667,0.715447,0.721311,0.710116,0.033790,10
1,0.002006,0.000632,0.010534,0.001230,3,{'n_neighbors': 3},0.707317,0.739837,0.715447,0.731707,0.762295,0.731321,0.019289,9
2,0.001675,0.000847,0.009308,0.001842,5,{'n_neighbors': 5},0.707317,0.764228,0.772358,0.772358,0.754098,0.754072,0.024322,6
3,0.001505,0.000450,0.008773,0.000631,7,{'n_neighbors': 7},0.707317,0.796748,0.707317,0.764228,0.786885,0.752499,0.038369,8
4,0.001805,0.000515,0.009526,0.001304,9,{'n_neighbors': 9},0.707317,0.796748,0.699187,0.764228,0.795082,0.752512,0.041934,7


In [12]:
grid.drop([ 'params' ,'mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
            'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score', 'split4_test_score' ], axis=1, inplace=True)
grid.head()

,param_n_neighbors,mean_test_score,std_test_score,rank_test_score
0,1,0.710116,0.033790,10
1,3,0.731321,0.019289,9
2,5,0.754072,0.024322,6
3,7,0.752499,0.038369,8
4,9,0.752512,0.041934,7


In [13]:
grid.nlargest(5, 'mean_test_score')

,param_n_neighbors,mean_test_score,std_test_score,rank_test_score
8,17,0.770385,0.023535,1
5,11,0.768799,0.041645,2
9,19,0.767106,0.028851,3
6,13,0.754138,0.043737,4
7,15,0.754112,0.037332,5


In [14]:
px.line(grid, x='param_n_neighbors', y='mean_test_score', title='KNN Classifier',width=800, height=600, markers=True)